In [1]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

In [2]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-3.5-flash-lite",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=2500,
    streaming=True,
)

In [4]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [5]:
from langchain.agents import create_agent
from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

# print("Running create_agent...", flush=True)
# agent_result = agent.invoke(
#     {"messages": [{"role": "user", "content": content}]},
#     config={"configurable": {"thread_id": "great-gatsby-lc"}},
# )
print("Running create_deep_agent...", flush=True)
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
# print("\ncreate_agent:")
# print(agent_result["messages"][-1].content_blocks)
print("\ncreate_deep_agent:")
print(deep_agent_result["messages"][-1].content_blocks)

Running create_deep_agent...


/home/gauravgupta/codebase/agentic-ai-krish-naik/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/gauravgupta/codebase/agentic-ai-krish-naik/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/gauravgupta/codebase/agentic-ai-krish-naik/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/home/gauravgupta/codebase/agentic-ai-krish-naik/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_


create_deep_agent:
[{'type': 'text', 'text': "Based on verified analysis of the Project Gutenberg plain-text file for F. Scott Fitzgerald's *The Great Gatsby* (URL: `https://www.gutenberg.org/files/64317/64317-0.txt`), here are the exact answers:\n\n1) **304** lines in the complete Gutenberg file contain the substring `Gatsby`.\n2) **181** is the 1-based line number of the first line in the file that contains `Daisy` (found in the sentence: *“Buchanans. Daisy was my second cousin once removed, and I’d known Tom”*).\n3) Set in the summer of 1922 on Long Island, the novel follows narrator Nick Carraway as he is drawn into the lavish, mysterious world of his wealthy neighbor, Jay Gatsby, and his obsessive quest to reunite with his former love, Daisy Buchanan. As Gatsby’s extravagant parties and illusion-driven ambitions collide with the callous reality of East Egg society, the story builds toward a tragic exploration of class, disillusionment, and the American Dream.", 'index': 0, 'extra